<a href="https://colab.research.google.com/github/Tom-Jung/Tom-Jung/blob/main/Stock_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from bs4 import BeautifulSoup
import time
from datetime import datetime
import pytz
from IPython.display import display, clear_output, HTML  # Jupyter Notebook 출력 관리

# 🔹 한국 시간대 설정
kst = pytz.timezone("Asia/Seoul")

# 🔹 네이버 금융에서 실시간 주가 및 전일 종가 가져오기
def get_naver_stock_data(stock_code):
    url = f"https://finance.naver.com/item/main.naver?code={stock_code}"
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # 현재가 가져오기
        price_tag = soup.select_one("p.no_today span.blind")
        current_price = int(price_tag.text.replace(",", "")) if price_tag else None

        # 전일 종가 가져오기
        prev_price_tag = soup.select("td.first span.blind")
        prev_close_price = int(prev_price_tag[0].text.replace(",", "")) if prev_price_tag else None

        return current_price, prev_close_price
    except requests.RequestException:
        return None, None

# 🔹 보유 주식 정보 1
portfolio_1 = [
    {"종목명": "코스모신소재", "종목코드": "005070", "매입가": 138329, "수량": 126},
    {"종목명": "POSCO홀딩스", "종목코드": "005490", "매입가": 458966, "수량": 145},
    {"종목명": "한화오션", "종목코드": "042660", "매입가": 58794, "수량": 400},
    {"종목명": "에코프로비엠", "종목코드": "247540", "매입가": 404714, "수량": 65},
    {"종목명": "RFHIC", "종목코드": "218410", "매입가": 30500, "수량": 100}
]

# 🔹 보유 주식 정보 2
portfolio_2 = [
    {"종목명": "코스모신소재", "종목코드": "005070", "매입가": 164598, "수량": 150},
    {"종목명": "POSCO홀딩스", "종목코드": "005490", "매입가": 522538, "수량": 130},
    {"종목명": "한화오션", "종목코드": "042660", "매입가": 44000, "수량": 300},
    {"종목명": "실리콘투", "종목코드": "257720", "매입가": 56300, "수량": 150},
    {"종목명": "RFHIC", "종목코드": "218410", "매입가": 104976, "수량": 50},
    {"종목명": "RFHIC", "종목코드": "462330", "매입가": 2040, "수량": 2000}
]

# 🔹 개별 포트폴리오 HTML 생성 함수
def generate_portfolio_html(title, portfolio):
    html = f"<h4 style='margin-top: 20px;'>{title}</h4>"
    html += """
    <table border="1" style="border-collapse: collapse; width: 100%; text-align: center; margin-bottom: 20px;">
        <tr style="background-color: #f2f2f2;">
            <th>종목명</th>
            <th>매입가</th>
            <th>전일 종가</th>
            <th>현재가</th>
            <th>수량</th>
            <th>현재가×수량</th>
            <th>손익(원)</th>
            <th>수익률(%)</th>
            <th>전일 대비(%)</th>
            <th>오늘 손익</th>
            <th>비중(%)</th>
        </tr>
    """

    total_purchase = 0
    total_current = 0
    total_today_profit = 0
    stock_data = []

    for stock in portfolio:
        current_price, prev_close_price = get_naver_stock_data(stock["종목코드"])
        if current_price is None or prev_close_price is None:
            continue

        purchase_price = stock["매입가"]
        quantity = stock["수량"]

        stock_total_purchase = purchase_price * quantity
        stock_total_current = current_price * quantity
        profit_loss = stock_total_current - stock_total_purchase
        profit_percent = (profit_loss / stock_total_purchase) * 100 if stock_total_purchase > 0 else 0

        day_change_percent = ((current_price - prev_close_price) / prev_close_price) * 100
        today_profit = (current_price - prev_close_price) * quantity

        total_purchase += stock_total_purchase
        total_current += stock_total_current
        total_today_profit += today_profit

        stock_data.append({
            "종목명": stock["종목명"], "매입가": purchase_price, "전일 종가": prev_close_price,
            "현재가": current_price, "수량": quantity, "현재가×수량": stock_total_current,
            "손익": profit_loss, "수익률": profit_percent, "전일 대비": day_change_percent,
            "오늘 손익": today_profit, "평가금액": stock_total_current
        })

    for data in stock_data:
        stock_weight = (data["평가금액"] / total_current) * 100 if total_current > 0 else 0

        # 색상 적용
        profit_color = "red" if data["손익"] >= 0 else "blue"
        percent_color = "red" if data["수익률"] >= 0 else "blue"
        day_change_color = "red" if data["전일 대비"] >= 0 else "blue"
        today_profit_color = "red" if data["오늘 손익"] >= 0 else "blue"
        weight_color = "green" if stock_weight >= 25 else "black"

        html += f"""
        <tr>
            <td>{data["종목명"]}</td>
            <td>{data["매입가"]:,}</td>
            <td>{data["전일 종가"]:,}</td>
            <td>{data["현재가"]:,}</td>
            <td>{data["수량"]:,}</td>
            <td>{data["현재가×수량"]:,}</td>
            <td style="color: {profit_color};">{data["손익"]:+,}</td>
            <td style="color: {percent_color};">{data["수익률"]:+.2f}%</td>
            <td style="color: {day_change_color};">{data["전일 대비"]:+.2f}%</td>
            <td style="color: {today_profit_color};">{data["오늘 손익"]:+,}</td>
            <td style="color: {weight_color};">{stock_weight:.2f}%</td>
        </tr>
        """

    total_profit_loss = total_current - total_purchase
    total_profit_percent = (total_profit_loss / total_purchase) * 100 if total_purchase != 0 else 0

    total_profit_color = "red" if total_profit_loss >= 0 else "blue"
    total_percent_color = "red" if total_profit_percent >= 0 else "blue"
    total_today_profit_color = "red" if total_today_profit >= 0 else "blue"

    html += f"""
    <tr style="font-weight: bold; background-color: #e6e6e6;">
        <td colspan="5">총합</td>
        <td>{total_current:,}</td>
        <td style="color: {total_profit_color};">{total_profit_loss:+,}</td>
        <td style="color: {total_percent_color};">{total_profit_percent:+.2f}%</td>
        <td>-</td>
        <td style="color: {total_today_profit_color};">{total_today_profit:+,}</td>
        <td>100.00%</td>
    </tr>
    </table>
    """
    return html

# 🔹 실시간 모니터링 실행
try:
    while True:
        clear_output(wait=True)  # 이전 출력 삭제

        current_time = datetime.now(kst).strftime("[%Y-%m-%d %H:%M:%S]")
        final_output = f"<h3>🕒 업데이트 시간: {current_time}</h3>"

        # 각각의 포트폴리오 HTML을 생성하여 병합
        final_output += generate_portfolio_html("📊 Me 포트폴리오", portfolio_1)
        final_output += generate_portfolio_html("📊 Pama 번째 포트폴리오", portfolio_2)

        # 결과 출력
        display(HTML(final_output))

        time.sleep(2)  # 2초마다 업데이트

except KeyboardInterrupt:
    print("\n📌 모니터링 프로그램이 종료되었습니다.")


📌 모니터링 프로그램이 종료되었습니다.
